In [10]:
!pip install pycryptodome ecdsa base58 tqdm

import hashlib
import binascii
from Crypto.Hash import SHA256, RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import random
from tqdm import tqdm
from multiprocessing import Pool
import sys

TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
START_PRIV = "f77dd3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03ca638371c4aa0"
START_ADDR = "17RPdVRKAHTyywX4aYMhfWdoTgeENsfguF"
ITERATIONS = 100_000_000
BATCH_SIZE = 10_000

def derive_address(priv_hex):
    try:
        priv_bytes = binascii.unhexlify(priv_hex)
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\x04' + sk.get_verifying_key().to_string()
        sha = SHA256.new(pub).digest()
        ripemd = RIPEMD160.new(sha).digest()
        payload = b'\x00' + ripemd
        checksum = SHA256.new(SHA256.new(payload).digest()).digest()[:4]
        return base58.b58encode(payload + checksum).decode()
    except:
        return "Error"

def hamming_distance(s1, s2):
    s1 = (s1 + '\0' * 34)[:34]
    s2 = (s2 + '\0' * 34)[:34]
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def flip_bits(priv_hex, num_flips):
    priv_int = int(priv_hex, 16)
    for _ in range(num_flips):
        bit_pos = random.randint(0, 255)
        priv_int ^= (1 << bit_pos)
    priv_int %= 2**256
    return f"{priv_int:064x}"

def process_batch(batch_start):
    best_priv = START_PRIV
    best_addr = START_ADDR
    best_hamm = hamming_distance(START_ADDR, TARGET_ADDRESS)

    for i in range(batch_start, batch_start + BATCH_SIZE):
        flips = random.randint(1, 5)
        new_priv = flip_bits(best_priv, flips)
        new_addr = derive_address(new_priv)
        if new_addr == "Error":
            continue
        new_hamm = hamming_distance(new_addr, TARGET_ADDRESS)

        if new_hamm < best_hamm:
            best_priv = new_priv
            best_addr = new_addr
            best_hamm = new_hamm
            print(f"Batch {batch_start // BATCH_SIZE}: Priv: {best_priv}, Addr: {best_addr}, Hamming: {best_hamm}")

        if new_addr == TARGET_ADDRESS:
            return {"priv": new_priv, "addr": new_addr, "hamming": 0, "jackpot": True}

    return {"priv": best_priv, "addr": best_addr, "hamming": best_hamm, "jackpot": False}

start_hamm = hamming_distance(START_ADDR, TARGET_ADDRESS)
print(f"Starting Priv: {START_PRIV}")
print(f"Starting Addr: {START_ADDR}")
print(f"Starting Hamming: {start_hamm}")

num_cores = 8  # Pro+ confirmed
print(f"Using {num_cores} CPU cores")
best_overall_priv = START_PRIV
best_overall_addr = START_ADDR
best_overall_hamm = start_hamm

with Pool(processes=num_cores) as pool:
    batches = range(0, ITERATIONS, BATCH_SIZE)
    results = list(tqdm(pool.imap_unordered(process_batch, batches), total=len(batches), desc="Bit-Flipping Batches"))

    for result in results:
        if result["jackpot"]:
            print(f"\n!!! JACKPOT !!! Priv: {result['priv']}, Addr: {result['addr']}")
            with open("jackpot.txt", "w") as f:
                f.write(f"Priv: {result['priv']}\nAddr: {result['addr']}")
            sys.exit(0)
        if result["hamming"] < best_overall_hamm:
            best_overall_priv = result["priv"]
            best_overall_addr = result["addr"]
            best_overall_hamm = result["hamming"]
            print(f"New Best: Priv: {best_overall_priv}, Addr: {best_overall_addr}, Hamming: {best_overall_hamm}")

print(f"\nBest Result: Priv: {best_overall_priv}, Addr: {best_overall_addr}, Hamming: {best_overall_hamm}")
with open("bitflip_results_rng_af.txt", "w") as f:
    f.write(f"Best Priv: {best_overall_priv}\nBest Addr: {best_overall_addr}\nBest Hamming: {best_overall_hamm}")

print(f"Python {sys.version}")

Starting Priv: f77dd3a23db5e0ad8a13827e8b07d923ea413990a06217a4b03ca638371c4aa0
Starting Addr: 17RPdVRKAHTyywX4aYMhfWdoTgeENsfguF
Starting Hamming: 25
Using 8 CPU cores


Bit-Flipping Batches:  36%|███▌      | 3566/10000 [1:50:23<3:19:10,  1.86s/it]


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')